<a href="https://colab.research.google.com/github/nischaya25/Salary-Prediction-using-Ensemble-Learning/blob/main/Intrusion_Detection_for_5G_Networks%E2%80%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas numpy scikit-learn tensorflow matplotlib seaborn flask joblib

In [2]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [4]:
df = pd.read_csv("/cleaned_intrusion_dataset.csv")

print(df.head())

   feature_0  feature_1  feature_2  feature_3  feature_4  feature_5  \
0  -2.565268  -0.472785   5.632443   2.002531  -0.905071  -1.601131   
1  -5.016748  -0.652570  -1.807753  -0.076741  -0.976327  -3.750632   
2   4.265028   1.414568  -1.812577  -2.316262  -2.625694  -0.124147   
3   5.316339   0.295956  28.947593  -0.040459  -6.684577   0.152203   
4   7.438659  -1.313327   7.673740   2.256872  -3.856906  -3.004590   

   feature_6  feature_7  feature_8  feature_9  ...  feature_21  feature_22  \
0  -4.323598  -4.979045   0.954298  -0.949713  ...   -1.521917   -2.574007   
1  -0.322719   0.929847   0.145276  -0.809447  ...   -0.309897   -1.171888   
2  -3.483063  -2.563455  -0.030792   1.218929  ...    2.559456    0.417948   
3   1.777875   1.182769   0.301609  -3.176431  ...    6.298994    6.432768   
4   2.569723  -1.937835   0.104488   2.716901  ...    4.188761    1.244554   

   feature_23  feature_24  feature_25  feature_26  feature_27  feature_28  \
0    8.341135   -3.687656  

In [5]:
df.fillna(df.mean(numeric_only=True), inplace=True)

In [6]:
df = pd.get_dummies(df)
print("After encoding:", df.shape)

After encoding: (10000, 31)


In [7]:
if "label" not in df.columns:
    raise ValueError("❌ 'label' column not found in dataset")
X = df.drop("label", axis=1)
y = df["label"]

In [8]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

print("Final Feature Shape:", X_scaled.shape)


Final Feature Shape: (10000, 30)


**training the model (randomforest)**

In [9]:
X = df.drop("label", axis=1)
y = df["label"]

In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [12]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Predictions
y_pred = rf.predict(X_test)

In [13]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.95
              precision    recall  f1-score   support

           0       0.95      0.95      0.95      1000
           1       0.95      0.95      0.95      1000

    accuracy                           0.95      2000
   macro avg       0.95      0.95      0.95      2000
weighted avg       0.95      0.95      0.95      2000



In [14]:
import joblib
import os

# Create the 'models' directory if it doesn't exist
os.makedirs("models", exist_ok=True)

joblib.dump(rf, "models/random_forest.pkl")

print("✅ Random Forest model saved!")

✅ Random Forest model saved!


In [15]:
from google.colab import files
files.download("models/random_forest.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

training **Autoencoder** (Anomaly Detection) train_autoencoder.py

---



In [16]:
X = df.drop("label", axis=1)
y = df["label"]


In [17]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

In [23]:
X_normal = X_scaled[y == 0]
print("Normal data shape:", X_normal.shape)

Normal data shape: (5002, 30)


In [25]:
import numpy as np
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.losses import MeanSquaredError

# Define the Autoencoder model
input_dim = X_normal.shape[1]
latent_dim = 16 # You can adjust this for more or less compression

autoencoder = Sequential([
    Input(shape=(input_dim,)), # Explicit Input layer to address the warning
    Dense(units=latent_dim * 2, activation='relu'),
    Dense(units=latent_dim, activation='relu'),
    Dense(units=latent_dim * 2, activation='relu'),
    Dense(units=input_dim, activation='sigmoid')
])

autoencoder.compile(optimizer='adam', loss='mse')

# Train the Autoencoder on normal data
# You might want to adjust epochs and batch_size based on your data
history = autoencoder.fit(
    X_normal, X_normal,
    epochs=50,
    batch_size=64,
    shuffle=True,
    validation_split=0.2
)

# Now, compute reconstruction error for the entire dataset
X_reconstructed = autoencoder.predict(X_scaled)
reconstruction_error = np.mean((X_scaled - X_reconstructed)**2, axis=1)

Epoch 1/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0173 - val_loss: 0.0162
Epoch 2/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0144 - val_loss: 0.0130
Epoch 3/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0119 - val_loss: 0.0110
Epoch 4/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0104 - val_loss: 0.0097
Epoch 5/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0091 - val_loss: 0.0087
Epoch 6/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0082 - val_loss: 0.0080
Epoch 7/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0075 - val_loss: 0.0072
Epoch 8/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0068 - val_loss: 0.0067
Epoch 9/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0063 - val_loss: 0.0063
Epoch 10/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0060 - val_loss: 0.0060
Epoch 11/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0057 - val_loss: 0.0057
Epoch 12/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0054 - val_lo

In [27]:
autoencoder.save("models/autoencoder.h5")

print("✅ Autoencoder model saved!")

✅ Autoencoder model saved!


In [28]:
from google.colab import files
files.download("models/autoencoder.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

HYBRID MODEL (RF + AUTOENCODER)

In [31]:
import joblib
from tensorflow.keras.models import load_model
from tensorflow.keras.losses import MeanSquaredError

rf = joblib.load("/random_forest.pkl")
autoencoder = load_model("/autoencoder.h5", custom_objects={'mse': MeanSquaredError()})

In [33]:
df = pd.read_csv("/cleaned_intrusion_dataset.csv")
X = df.drop("label", axis=1)

In [34]:
scaler = MinMaxScaler()
scaler.fit(X)

MinMaxScaler()

In [35]:
def hybrid_predict(input_data, w1=0.6, w2=0.4, threshold=0.5):

    # Step 1: Scale input
    input_scaled = scaler.transform(input_data)

    # Step 2: Random Forest probability (known attack)
    rf_prob = rf.predict_proba(input_data)[:, 1]

    # Step 3: Autoencoder reconstruction error (anomaly)
    reconstructed = autoencoder.predict(input_scaled)
    error = np.mean(np.square(input_scaled - reconstructed), axis=1)

    # Normalize error (0–1)
    error = (error - error.min()) / (error.max() - error.min() + 1e-8)

    # Step 4: Weighted Fusion
    final_score = (w1 * rf_prob) + (w2 * error)

    # Step 5: Final decision
    prediction = (final_score > threshold).astype(int)
    return prediction, final_score

In [37]:
import pandas as pd

# Load dataset
df = pd.read_csv("/cleaned_intrusion_dataset.csv")

X = df.drop("label", axis=1)

# Take sample data
sample = X.iloc[:10]

pred, score = hybrid_predict(sample)

print("Predictions:", pred)
print("Scores:", score)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
Predictions: [1 1 0 0 0 0 0 1 1 0]
Scores: [0.88242702 0.73033431 0.16878223 0.075      0.19995282 0.23463702
 0.29436776 0.51099962 0.78501617 0.18960562]


In [38]:
#Tune weights:
w1 = 0.7
w2 = 0.3

In [39]:
#Adjust threshold:
threshold = 0.5  # or 0.6 for stricter detection